# Agrupamentos historicos que antecedem a calibracao

**Papel nesta entrega:** Suplementar: analise historica descrita no trabalho

**Origem:** https://colab.research.google.com/drive/1T60tj8Q6um5FbEuprwZCNvcxkqpxfzYg

Copia de publicacao organizada em 7 out. 2026. Saidas, contadores e metadados de sessao foram removidos; isso nao significa que a campanha original nao tenha sido executada. Nenhum resultado novo foi produzido na curadoria. O registro de proveniencia relaciona codigo, fonte e alteracoes de configuracao.

Dependencias e caminhos sao os da campanha historica, nao das quatro execucoes padronizadas. Esses cadernos nao devem substituir os resultados principais. Consulte o mapa de cobertura e mantenha separadas as datas de execucao. 

# TCC | Etapa 04 - Aprendizado nao supervisionado dos perfis multibloco
**Mario Costa Cruz | EUbOPEN / IMTM HepG2**

**Objetivo:** descobrir grupos de perfis e avaliar sua estabilidade, sem depender de anotacoes farmacologicas e sem criar uma classe positiva com os tres candidatos que repetiram a regra.

Esta etapa ajusta modelos **K-means**, mas nao classificadores supervisionados. Os grupos sao particoes matematicas exploratorias; nao sao automaticamente mecanismos, organelas afetadas ou hits confirmados.

A entrada e a tabela `completas.csv` da etapa 03: 2447 condicoes completas. Somente quatro canais (AGP, DNA, ER, Mito) entram no ajuste. Os tres blocos morfologicos sao apresentados como contexto. Os 12 candidatos originais sao localizados nos grupos **depois** do ajuste, sem participar da escolha de k. Nenhum arquivo R4 e lido.

**Execute todas as celulas na ordem.** As saidas vao para uma pasta nova, sem sobrescrever nenhuma etapa anterior.

## 1. Ambiente e configuracao
A tabela tem aproximadamente 2 MB. Polars e usado na leitura quando disponivel; pandas e a alternativa de leitura para essa tabela pequena. O uso de uma ou outra biblioteca e registrado. Nao e necessario carregar ou repetir a selecao de 735 medidas.

In [ ]:
from pathlib import Path
import os,sys,importlib.util,subprocess
for module,package in [('numpy','numpy'),('pandas','pandas'),('sklearn','scikit-learn'),('matplotlib','matplotlib'),('threadpoolctl','threadpoolctl')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install',package])
try:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    IN_COLAB=True
except ImportError:
    IN_COLAB=False
from IPython.display import display,Markdown,Image
RAIZ=Path(os.environ.get('TCC_RAIZ','/content/drive/MyDrive/mba_tcc'))


In [ ]:
"""Etapa 04: K-means exploratorio, sem rotulos farmacologicos e sem R4.
A avaliacao e de estabilidade CONDICIONAL a representacao predefinida.
Nao e validacao independente nem descoberta de mecanismos biologicos.
"""
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import warnings
import numpy as np
import pandas as pd
try:
    import polars as pl
except ImportError:
    pl = None
import sklearn
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score, adjusted_mutual_info_score
from sklearn.metrics.pairwise import pairwise_distances
from threadpoolctl import threadpool_limits
import matplotlib.pyplot as plt

SOURCE_SHA256 = 'de4566acaa64208b5e41294ac9e4acc506833a23541ae65f5c0f383fa05bddcf'
SOURCE_REL = 'resultados_tcc/mapa_multibloco/R1_R3_20260926T152153_537184Z/tabelas/completas.csv'
SOURCE_URL = 'https://drive.google.com/file/d/1vB1LoU1oZQg1CxjYqKbRClxhfsNblU2P/view'
P, E, D = 'Metadata_Plate_ID', 'Metadata_EOS', 'Metadata_Concentration'
KEY = [P, E, '_dose_key']
REPS = ['R1', 'R2', 'R3']
CHANNELS = ['AGP', 'DNA', 'ER', 'Mito']
BLOCKS = CHANNELS + ['Morfologia_Cells', 'Morfologia_Cyto', 'Morfologia_Nuc']
SPACES = ['magnitude_4canais', 'contrastes_4canais']
DEFAULT_CONFIG = dict(seed=20260926, ks=list(range(2,9)), n_init=30, max_iter=500,
                      n_subsamples=20, fraction_compounds=0.8, n_shuffle=200,
                      silhouette_max_n=3000)


def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1048576), b''): h.update(chunk)
    return h.hexdigest()


def as_bool(s):
    b=s.astype(str).str.lower().str.strip().map({'true':True,'false':False,'1':True,'0':False})
    if b.isna().any(): raise ValueError('Booleano invalido na fonte.')
    return b.astype(bool)


def validate_data(df, expected_n=None):
    df=df.copy()
    required=KEY+[D,'completa_3rep_7blocos','prioridade_R1_R3_original']
    required += [f'A_{b}_{r}' for b in BLOCKS for r in REPS]
    required += [f'A_{b}_mediana' for b in BLOCKS]
    missing=set(required)-set(df)
    if missing: raise ValueError('Colunas ausentes: '+str(sorted(missing)))
    if any('R4' in c for c in df): raise ValueError('A fonte nao pode conter resultados R4.')
    if expected_n is not None and len(df)!=expected_n:
        raise ValueError(f'Esperadas {expected_n} condicoes; recebidas {len(df)}.')
    if df[KEY].isna().any().any() or df.duplicated(KEY).any():
        raise ValueError('Chaves nulas/duplicadas: nenhuma agregacao silenciosa.')
    for c in KEY:
        df[c]=df[c].astype(str).str.strip()
        if df[c].eq('').any(): raise ValueError('Chave vazia.')
    df=df.sort_values(KEY,kind='stable').reset_index(drop=True)
    df['completa_3rep_7blocos']=as_bool(df['completa_3rep_7blocos'])
    df['prioridade_R1_R3_original']=as_bool(df['prioridade_R1_R3_original'])
    if not df['completa_3rep_7blocos'].all(): raise ValueError('Fonte inclui condicoes incompletas.')
    cube=np.stack([df[[f'A_{b}_{r}' for b in BLOCKS]].to_numpy(float) for r in REPS],axis=1)
    if not np.isfinite(cube).all() or (cube<0).any():
        raise ValueError('Escores negativos/nao finitos. Nao ha imputacao nesta etapa.')
    stored=df[[f'A_{b}_mediana' for b in BLOCKS]].to_numpy(float)
    if not np.allclose(np.median(cube,axis=1),stored,rtol=1e-9,atol=1e-10):
        raise ValueError('Medianas nao reproduzem as tres replicas.')
    return df,cube



In [ ]:
CONFIG=dict(DEFAULT_CONFIG)
FONTE=RAIZ/SOURCE_REL
if not FONTE.is_file(): raise FileNotFoundError(f'Fonte da etapa 03 nao encontrada: {FONTE}')
if sha256(FONTE)!=SOURCE_SHA256:
    raise RuntimeError('Fonte diferente da auditada. Nao substituir automaticamente pelo arquivo mais recente.')
if pl is not None:
    data=pl.read_csv(FONTE,schema_overrides={D:pl.String},infer_schema_length=10000)
    df=pd.DataFrame(data.to_dict(as_series=False)); LEITOR='polars'
else:
    df=pd.read_csv(FONTE,dtype={D:str}); LEITOR='pandas'
df,cube=validate_data(df,expected_n=2447)
if int(df['prioridade_R1_R3_original'].sum())!=12:
    raise RuntimeError('A lista original de prioridades divergiu.')
RUN=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT=RAIZ/'resultados_tcc'/'agrupamento_nao_supervisionado'/('R1_R3_'+RUN)
OUT.mkdir(parents=True,exist_ok=False)
PROVENANCE=dict(execucao=RUN,ambiente='Colab' if IN_COLAB else 'Local',
 fonte=str(FONTE),fonte_drive=SOURCE_URL,sha256=SOURCE_SHA256,leitor=LEITOR,
 config=CONFIG,natureza='Exploratoria pos-hoc; estabilidade condicional',
 escopo='Quatro canais; morfologia e prioridades apenas contexto. Sem R4 e sem rotulos farmacologicos.')
(OUT/'plano_antes_ajuste.json').write_text(json.dumps(PROVENANCE,indent=2,ensure_ascii=False),encoding='utf-8')
print('Condicoes:',len(df),'| Compostos:',df[ E ].nunique(),'| Leitor:',LEITOR)
print('Pasta de saida:',OUT)


## 2. Duas representacoes e uma pergunta de estabilidade
**Magnitude:** `log(1+A)` dos quatro canais, seguido de centralizacao pela mediana e divisao pelo intervalo interquartil.

**Contrastes relativos:** subtrair a media dos quatro valores `log(1+A)` de cada condicao, antes da mesma escala robusta por canal. Isso retira o componente comum nessa escala; nao converte canais em unidades biologicas equivalentes e nao garante seletividade. Perto dos controles, contrastes relativos podem refletir variabilidade tecnica.

Para cada representacao, avaliar k de 2 a 8, com 30 inicializacoes K-means++ e semente registrada. Todos os tamanhos de grupos sao mostrados, sem eliminar grupos pequenos. A solucao de referencia maximiza silhouette interno (empate: menor k). **Esta escolha nao demonstra um numero verdadeiro de fenotipos.** K-means pode particionar um continuum.

**Estabilidade por subamostragem:** 20 subconjuntos de 80% dos compostos, com todas as condicoes de cada composto mantidas juntas. Reajustar escala e K-means; comparar atribuicoes com a particao de referencia. O ARI de compostos fora da subamostra tambem e relativo a essa referencia ajustada ao conjunto completo, portanto nao e desempenho independente.

**Estabilidade entre replicas:** ajustar escala e K-means na mediana de duas replicas e aplicar a terceira, em tres rotacoes. As condicoes continuam sendo as mesmas. As referencias P95 e a selecao original ja utilizaram R1-R3; esta rotacao nao refaz a validacao de todo o pipeline.

Silhouette e ARI nao sao acuracia. A regra original A>1 nao e modificada.

In [ ]:

def representation(raw, space):
    raw=np.asarray(raw,float)
    if raw.ndim!=2 or raw.shape[1]!=4 or not np.isfinite(raw).all() or (raw<0).any():
        raise ValueError('Esperada matriz finita de quatro canais, sem valores negativos.')
    log=np.log1p(raw)
    if space=='magnitude_4canais': return log
    if space=='contrastes_4canais': return log-log.mean(axis=1,keepdims=True)
    raise ValueError('Representacao nao conhecida: '+space)


def fit_model(raw, space, k, seed, cfg):
    base=representation(raw,space)
    if np.unique(base,axis=0).shape[0]<k:
        raise ValueError('Menos perfis distintos do que grupos solicitados.')
    scaler=RobustScaler(quantile_range=(25,75))
    x=scaler.fit_transform(base)
    model=KMeans(n_clusters=int(k),init='k-means++',n_init=int(cfg['n_init']),
                 max_iter=int(cfg['max_iter']),random_state=int(seed),algorithm='lloyd')
    model.fit(x)
    if len(np.unique(model.labels_))!=k: raise ValueError('Particao degenerada.')
    return scaler,model,x


def predict(raw, space, scaler, model):
    return model.predict(scaler.transform(representation(raw,space)))


def group_draws(compounds,cfg):
    unique=np.array(sorted(set(compounds)))
    if len(unique)<4: raise ValueError('Poucos compostos para avaliar subamostragem.')
    size=min(len(unique)-1,max(2,int(np.ceil(cfg['fraction_compounds']*len(unique)))))
    rng=np.random.default_rng(cfg['seed'])
    return [np.isin(compounds,rng.choice(unique,size=size,replace=False)) for _ in range(cfg['n_subsamples'])]


def eta_squared(values, labels):
    values=np.asarray(values,float); total=np.sum((values-values.mean())**2)
    if total<=0: return np.nan
    within=sum(np.sum((values[labels==g]-values[labels==g].mean())**2) for g in np.unique(labels))
    return float(max(0,1-within/total))


def run_models(df,cube,cfg=None,verbose=True):
    cfg=dict(DEFAULT_CONFIG if cfg is None else cfg)
    raw=np.median(cube[:,:,:4],axis=1)
    compound=df[E].to_numpy(); plate=df[P].to_numpy()
    draws=group_draws(compound,cfg)
    candidate_rows=[]; sub_rows=[]; rotation_rows=[]; models={}
    global_strength=np.log1p(raw).mean(axis=1)
    with threadpool_limits(limits=1):
        for si,space in enumerate(SPACES):
            base=representation(raw,space)
            scaler0=RobustScaler().fit(base)
            x0=scaler0.transform(base)
            # Distancias completas para silhouette: a fonte atual tem apenas 2447 linhas.
            sample=np.arange(len(df))
            if len(sample)>cfg['silhouette_max_n']:
                sample=np.sort(np.random.default_rng(cfg['seed']).choice(len(df),cfg['silhouette_max_n'],replace=False))
            distances=pairwise_distances(x0[sample])
            np.fill_diagonal(distances,0)
            for k in cfg['ks']:
                seed=cfg['seed']+1000*si+int(k)
                try: scaler,model,x=fit_model(raw,space,k,seed,cfg)
                except ValueError as ex:
                    candidate_rows.append(dict(espaco=space,k=k,status=str(ex)));continue
                labels=model.labels_; sizes=np.bincount(labels,minlength=k)
                sil=np.nan
                if 1<len(np.unique(labels[sample]))<len(sample):
                    sil=float(silhouette_score(distances,labels[sample],metric='precomputed'))
                row=dict(espaco=space,k=k,status='ok',silhouette=sil,n_silhouette=len(sample),
                         inercia=float(model.inertia_),n_menor_grupo=int(sizes.min()),
                         n_maior_grupo=int(sizes.max()),
                         eta2_magnitude_global=eta_squared(global_strength,labels),
                         AMI_placa=float(adjusted_mutual_info_score(plate,labels)))
                for s,mask in enumerate(draws):
                    sc,km,_=fit_model(raw[mask],space,k,seed+10000+s,cfg)
                    y=predict(raw,space,sc,km)
                    ari=float(adjusted_rand_score(labels,y))
                    excluded=~mask
                    ari_ex=float(adjusted_rand_score(labels[excluded],y[excluded])) if excluded.sum()>1 else np.nan
                    sub_rows.append(dict(espaco=space,k=k,repeticao=s+1,n_condicoes_ajuste=int(mask.sum()),
                                         n_compostos_ajuste=len(np.unique(compound[mask])),
                                         ARI_todas=ari,ARI_compostos_fora=ari_ex))
                for r,test_rep in enumerate(REPS):
                    # As condicoes sao as mesmas; so uma replica e retirada do ajuste do K-means.
                    dev=np.median(cube[:,[j for j in range(3) if j!=r],:4],axis=1)
                    test=cube[:,r,:4]
                    sc,km,_=fit_model(dev,space,k,seed+20000+r,cfg)
                    train_labels=km.labels_; assigned=predict(test,space,sc,km)
                    rotation_rows.append(dict(espaco=space,k=k,replica_aplicacao=test_rep,
                         replicas_ajuste='+'.join(q for q in REPS if q!=test_rep),
                         ARI=float(adjusted_rand_score(train_labels,assigned)),
                         pct_mesmo_grupo=float(100*np.mean(train_labels==assigned)),
                         n_grupos_aplicacao=len(np.unique(assigned))))
                models[(space,int(k))]=(scaler,model,x)
                candidate_rows.append(row)
                if verbose: print(f'{space} | k={k}: ajuste, {len(draws)} subamostras e 3 rotacoes concluidos.')
    candidates=pd.DataFrame(candidate_rows)
    subs=pd.DataFrame(sub_rows); rotations=pd.DataFrame(rotation_rows)
    ssum=subs.groupby(['espaco','k']).agg(ARI_sub_mediana=('ARI_todas','median'),
        ARI_sub_min=('ARI_todas','min'),ARI_fora_mediana=('ARI_compostos_fora','median')).reset_index()
    rsum=rotations.groupby(['espaco','k']).agg(ARI_rep_mediana=('ARI','median'),
        ARI_rep_min=('ARI','min'),mesmo_grupo_rep_media_pct=('pct_mesmo_grupo','mean')).reset_index()
    candidates=candidates.merge(ssum,on=['espaco','k'],how='left').merge(rsum,on=['espaco','k'],how='left')
    selected={}
    for space in SPACES:
        valid=candidates[(candidates.espaco==space)&candidates.status.eq('ok')&candidates.silhouette.notna()]
        if valid.empty: raise RuntimeError('Sem particao avaliavel no espaco '+space)
        # Regra operacional de apresentacao; nao estima numero verdadeiro de fenotipos.
        selected[space]=int(valid.sort_values(['silhouette','k'],ascending=[False,True]).iloc[0]['k'])
    return dict(candidatos=candidates,subamostras=subs,rotacoes=rotations,
                modelos=models,k_referencia=selected,config=cfg)



In [ ]:
RESULT=run_models(df,cube,CONFIG,verbose=True)
display(RESULT['candidatos'])
print('k de referencia para apresentacao:',RESULT['k_referencia'])


## 3. Interpretacao dos grupos, contexto e exportacao
Os IDs G01, G02 etc. sao numericos, ordenados por magnitude global apenas para facilitar a leitura. Nao recebem nomes biologicos automaticos.

Serao exportados perfis medianos e intervalos interquartis, distribuicao por placa, atribuicoes individuais e localizacao dos 12 candidatos. Uma referencia adicional permuta os rotulos de cada replica dentro de placa (200 vezes): e um diagnostico de correspondencia, nao um teste de existencia de clusters, valores-p ou FDR.

A fracao `eta2_magnitude_global` resume quanto da variacao de magnitude geral fica entre os grupos ajustados. Nao e um R2 preditivo. Uma particao explicada principalmente por magnitude global nao deve ser anunciada como tipos de efeito especificos de compartimentos.

In [ ]:

def describe_models(df,cube,result):
    cfg=result['config']; raw=np.median(cube,axis=1)
    assignments=df[KEY+[D,'prioridade_R1_R3_original']].copy()
    groups=[]; centroids=[]; plates=[]; shuffles=[]; similarities=[]
    for si,space in enumerate(SPACES):
        k=result['k_referencia'][space]
        scaler,model,x=result['modelos'][(space,k)]; labels=model.labels_
        # IDs G01..Gk ordenados apenas para leitura; nao sao classes biologicas.
        order=sorted(np.unique(labels),key=lambda g:(np.median(np.log1p(raw[labels==g,:4]).mean(1)),int(g)))
        rename={int(g):'G'+str(i+1).zfill(2) for i,g in enumerate(order)}
        assignments[space+'_grupo']=[rename[int(g)] for g in labels]
        rr=np.stack([predict(cube[:,r,:4],space,scaler,model) for r in range(3)],axis=1)
        assignments[space+'_3rep_mesmo_grupo']=(rr==rr[:,[0]]).all(1)
        for r in range(3): assignments[space+'_'+REPS[r]]=[rename[int(g)] for g in rr[:,r]]
        distances=model.transform(x)
        assignments[space+'_distancia_centro']=distances.min(1)
        for g in order:
            mask=labels==g
            item=dict(espaco=space,k=k,grupo=rename[int(g)],id_modelo=int(g),n_condicoes=int(mask.sum()),
                      n_compostos=df.loc[mask,E].nunique(),n_prioridades_originais=int(df.loc[mask,'prioridade_R1_R3_original'].sum()),
                      pct_3rep_mesmo_grupo=float(100*(rr[mask]==rr[mask,0][:,None]).all(1).mean()),
                      magnitude_global_mediana=float(np.median(np.log1p(raw[mask,:4]).mean(1))))
            for j,b in enumerate(BLOCKS):
                v=raw[mask,j]
                item['A_'+b+'_mediana']=float(np.median(v))
                item['pct_'+b+'_acima_2de3']=float(100*((cube[mask,:,j]>1).sum(1)>=2).mean())
                centroids.append(dict(espaco=space,k=k,grupo=rename[int(g)],bloco=b,
                                      papel='ajuste' if b in CHANNELS else 'contexto',mediana_A=float(np.median(v)),
                                      q25_A=float(np.quantile(v,.25)),q75_A=float(np.quantile(v,.75))))
            groups.append(item)
            for p in sorted(df[P].unique()):
                pm=df[P].eq(p).to_numpy()
                plates.append(dict(espaco=space,k=k,grupo=rename[int(g)],placa=p,n=int((mask&pm).sum()),
                                   pct_na_placa=float(100*(mask&pm).sum()/pm.sum())))
        # Referencia contextual: permuta correspondencia de rotulos dentro de placa.
        # Nao e teste de ausencia de grupos nem inferencia de FDR.
        rng=np.random.default_rng(cfg['seed']+50000+si)
        for r in range(3):
            observed=float(adjusted_rand_score(labels,rr[:,r]))
            null=[]
            for t in range(cfg['n_shuffle']):
                perm=rr[:,r].copy()
                for p in sorted(df[P].unique()):
                    inds=np.where(df[P].to_numpy()==p)[0]
                    perm[inds]=rng.permutation(perm[inds])
                null.append(float(adjusted_rand_score(labels,perm)))
            shuffles.append(dict(espaco=space,k=k,replica=REPS[r],ARI_correspondencia=observed,
                                 ARI_permutado_mediana=float(np.median(null)),ARI_permutado_P95=float(np.quantile(null,.95)),
                                 n_permutacoes=cfg['n_shuffle']))
        similarities.append(dict(espaco=space,k=k,AMI_placa=float(adjusted_mutual_info_score(df[P],labels)),
                                 eta2_magnitude_global=eta_squared(np.log1p(raw[:,:4]).mean(1),labels)))
    result.update(atribuicoes=assignments,grupos=pd.DataFrame(groups),perfis=pd.DataFrame(centroids),
                  grupos_por_placa=pd.DataFrame(plates),permutacoes=pd.DataFrame(shuffles),diagnosticos=pd.DataFrame(similarities))
    result['ARI_entre_representacoes']=float(adjusted_rand_score(assignments[SPACES[0]+'_grupo'],assignments[SPACES[1]+'_grupo']))
    return result


def figure_set(result,out):
    folder=Path(out)/'figuras'; folder.mkdir(exist_ok=True)
    captions=[]
    def save(fig,name,caption):
        fig.tight_layout()
        for ext in ['png','svg']: fig.savefig(folder/(name+'.'+ext),dpi=300,bbox_inches='tight')
        plt.close(fig); captions.append((name,caption))
    for si,space in enumerate(SPACES):
        desc='Magnitude' if si==0 else 'Contrastes relativos'
        metrics=result['candidatos'].query('espaco==@space').sort_values('k')
        fig,ax=plt.subplots(figsize=(7,4.5));ax.plot(metrics.k,metrics.silhouette,marker='o')
        ax.set(xlabel='Numero de grupos (k)',ylabel='Silhouette medio',title=desc+' | separacao interna')
        ax.set_xticks(metrics.k);save(fig,f'C{1+si*4:02d}_silhouette_{space}',
             'Silhouette interno no espaco indicado. Nao comparar como desempenho preditivo; o algoritmo pode particionar um continuum.')
        fig,ax=plt.subplots(figsize=(7,4.5))
        ax.plot(metrics.k,metrics.ARI_sub_mediana,marker='o',label='Subamostragem por composto: mediana')
        ax.plot(metrics.k,metrics.ARI_rep_mediana,marker='s',label='Ajuste 2 replicas / aplicacao 1: mediana')
        ax.set(xlabel='Numero de grupos (k)',ylabel='Indice de Rand ajustado (ARI)',title=desc+' | estabilidade condicional')
        ax.set_xticks(metrics.k);ax.legend(fontsize=8)
        save(fig,f'C{2+si*4:02d}_estabilidade_{space}',
             'Concordancia de particoes sob subamostragem e rotacao de replicas. Features e P95 anteriores permanecem fixos; nao e validacao independente.')
        k=result['k_referencia'][space];rows=result['perfis'].query('espaco==@space')
        fig,ax=plt.subplots(figsize=(8,4.8))
        for group,g in rows.groupby('grupo'):
            g=g.set_index('bloco').reindex(BLOCKS)
            ax.plot(range(7),g.mediana_A,marker='o',label=group)
        ax.axhline(1,ls='--',lw=.8);ax.axvline(3.5,ls=':',lw=.8)
        ax.set_xticks(range(7),['AGP','DNA','ER','Mito','Forma\nCells','Forma\nCyto','Forma\nNuc'])
        ax.set_yscale('log');ax.set(ylabel='Mediana de A por grupo (escala log)',title=f'{desc} | perfis dos grupos de referencia, k={k}')
        ax.legend(fontsize=8);save(fig,f'C{3+si*4:02d}_perfis_{space}',
             'Perfis medianos na escala original A. Quatro canais entram no ajuste; os tres blocos morfologicos sao apenas contexto. A linha A=1 e a referencia original.')
        fig,ax=plt.subplots(figsize=(8,4.8))
        table=result['grupos_por_placa'].query('espaco==@space').pivot(index='placa',columns='grupo',values='pct_na_placa')
        bottom=np.zeros(len(table))
        for group in table:
            ax.bar(table.index,table[group],bottom=bottom,label=group);bottom+=table[group].to_numpy()
        ax.set(ylabel='Condicoes por placa (%)',title=desc+' | distribuicao dos grupos por placa',ylim=(0,100))
        ax.legend(fontsize=8);save(fig,f'C{4+si*4:02d}_placas_{space}',
             'Composicao dos grupos por placa, com denominador especifico da placa. Composicao quimica e efeito tecnico nao sao separados por este diagnostico.')
    return captions


def report(result,n):
    lines=['# Agrupamento fenotipico nao supervisionado - R1-R3','',
           f'Condicoes: {n}. Quatro canais no ajuste; morfologia apenas como contexto.',
           'Nenhum rotulo farmacologico ou status R4 entrou no ajuste. Esta e uma etapa exploratoria pos-hoc.',
           'K de referencia maximiza silhouette interno entre 2 e 8 (empate: menor k); nao e numero verdadeiro de mecanismos.',
           'Nao ha limiar minimo de tamanho eliminando grupos raros; tamanhos e estabilidade sao apresentados integralmente.','']
    for space in SPACES:
        k=result['k_referencia'][space]
        row=result['candidatos'].query('espaco==@space and k==@k').iloc[0]
        groups=result['grupos'].query('espaco==@space')
        lines += [f'## {space}',f'K de referencia: {k}; silhouette: {row.silhouette:.4f}.',
          f'ARI mediano das subamostras: {row.ARI_sub_mediana:.4f}; ARI mediano das rotacoes: {row.ARI_rep_mediana:.4f}.',
          f'Fracao descritiva da variacao de magnitude global entre grupos (eta2): {row.eta2_magnitude_global:.4f}.',
          'Este eta2 descreve a particao ajustada, nao e R2 preditivo nem prova de causalidade.']
        for _,g in groups.iterrows():
            lines.append(f"{g.grupo}: {int(g.n_condicoes)} condicoes; {int(g.n_compostos)} compostos; {int(g.n_prioridades_originais)} prioridades originais; mesma atribuicao nas 3 replicas: {g.pct_3rep_mesmo_grupo:.1f}%.")
        lines += ['']
    lines += ['## Como interpretar e continuar',
        'A solucao de magnitude pode separar principalmente intensidade global. A solucao de contrastes remove a media de log(1+A) em cada condicao, mas nao isola seletividade biologica.',
        'Grupos de baixa magnitude podem conter contrastes relativos produzidos por variabilidade tecnica. Os escores A originais continuam necessarios para interpretacao.',
        'Silhouette e ARI nao sao acuracia. ARI alto sob subamostragem nao garante repeticao entre replicas.',
        'Todas as metricas sao condicionais a selecao de caracteristicas e referencias P95 ajustadas anteriormente em R1-R3. A rotacao nao reconstitui uma validacao independente do pipeline inteiro.',
        'A aplicacao em uma replica compara as mesmas condicoes, nao novos compostos.',
        'As permutacoes de rotulos dentro de placa sao referencia descritiva de correspondencia; nao sao teste de existencia de grupos nem controle de falsas descobertas.',
        'O K-means sempre tenta formar k grupos, mesmo que os dados sejam continuos; separacao, estabilidade, magnitude e placas devem ser interpretadas juntas.',
        'Os 12 candidatos originais aparecem somente depois do ajuste para contextualizacao, sem reclassificacao. Ausencia de um grupo mitocondrial exclusivo nao invalida candidatos raros.',
        'R4 nao foi lida. Nenhum classificador supervisionado foi treinado.',
        'Foram ajustados modelos de aprendizado nao supervisionado (K-means).']
    return '\n\n'.join(lines)


def export_all(result,df,cube,out,provenance):
    out=Path(out);tab=out/'tabelas';tab.mkdir(exist_ok=True)
    for name in ['candidatos','subamostras','rotacoes','atribuicoes','grupos','perfis','grupos_por_placa','permutacoes','diagnosticos']:
        result[name].to_csv(tab/(name+'.csv'),index=False)
    result['atribuicoes'].loc[result['atribuicoes'].prioridade_R1_R3_original].to_csv(tab/'localizacao_12_prioridades.csv',index=False)
    model_data=[]
    for (space,k),(scaler,model,x) in result['modelos'].items():
        model_data.append(dict(espaco=space,k=k,channels=CHANNELS,center=scaler.center_.tolist(),scale=scaler.scale_.tolist(),
             cluster_centers=model.cluster_centers_.tolist(),seed=model.random_state,
             formula='log1p(A); subtrair media da linha apenas em contrastes; (valor-center)/scale; centro mais proximo'))
    (out/'modelos_numericos.json').write_text(json.dumps(model_data,indent=2,ensure_ascii=False),encoding='utf-8')
    captions=figure_set(result,out)
    text=report(result,len(df));(out/'relatorio_agrupamento.md').write_text(text,encoding='utf-8')
    (out/'legendas.md').write_text('\n\n'.join('## '+n+'\n'+c for n,c in captions),encoding='utf-8')
    provenance.update(n_condicoes=len(df),n_compostos=df[E].nunique(),k_referencia=result['k_referencia'],
                      ARI_entre_representacoes=result['ARI_entre_representacoes'],n_figuras=len(captions),
                      numpy=np.__version__,pandas=pd.__version__,polars=(pl.__version__ if pl is not None else None),sklearn=sklearn.__version__,
                      python=platform.python_version())
    (out/'manifesto_agrupamento.json').write_text(json.dumps(provenance,indent=2,ensure_ascii=False),encoding='utf-8')
    paragraphs=['# Texto para o TCC - resultados locais desta execucao',
      'Foram ajustados modelos de K-means para '+str(len(df))+' condicoes completas de R1-R3, utilizando os quatro escores de canais em duas representacoes exploratorias. A primeira reteve a magnitude por canal apos log(1+A); a segunda subtraiu a media dos quatro canais nessa escala em cada condicao. Cada representacao foi centralizada e escalada por mediana e intervalo interquartil. Os tres blocos de morfologia foram utilizados somente para contextualizar os grupos.']
    for space in SPACES:
        k=result['k_referencia'][space];row=result['candidatos'].query('espaco==@space and k==@k').iloc[0]
        paragraphs.append(f'Na representacao {space}, a solucao de referencia apresentou k={k} e silhouette de {row.silhouette:.4f}. O ARI mediano de subamostragem foi {row.ARI_sub_mediana:.4f}, enquanto o ARI mediano sob rotacao das replicas foi {row.ARI_rep_mediana:.4f}. Esses valores descrevem estabilidade condicional a representacao previamente calculada, sem constituir validacao independente ou evidencia de mecanismos biologicos distintos.')
    (out/'texto_TCC_agrupamento.md').write_text('\n\n'.join(paragraphs),encoding='utf-8')
    return text,captions


In [ ]:
RESULT=describe_models(df,cube,RESULT)
TEXT,CAPTIONS=export_all(RESULT,df,cube,OUT,PROVENANCE)
if sha256(FONTE)!=SOURCE_SHA256: raise RuntimeError('A fonte mudou durante a execucao.')
import shutil
PACKAGE=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
display(RESULT['grupos'])
display(Markdown(TEXT))
print(f'CONCLUIDO: {len(CAPTIONS)} figuras; modelos K-means e tabelas exportados.')
print('Pasta:',OUT)
print('Pacote:',PACKAGE)
print('R4 nao lida. Nenhum classificador supervisionado. Nenhum candidato original reclassificado.')


## 4. Figuras
Cada grafico e salvo individualmente em PNG (300 dpi) e SVG. Perfis na escala A apresentam os quatro canais usados no ajuste e, apos a separacao vertical, a morfologia auxiliar. A linha A=1 e somente a referencia original.

In [ ]:
for name,caption in CAPTIONS:
    display(Markdown('### '+name+'\n'+caption))
    display(Image(filename=str(OUT/'figuras'/(name+'.png')),width=850))


## Rastreabilidade e limites
Este notebook foi testado com a tabela real da etapa 03 em um ambiente local do assistente. Essa verificacao nao executou sua sessao ativa do Colab. Os resultados apresentados nesta execucao sao calculados de novo a partir da fonte conferida por SHA-256.

O manifesto registra versoes, fonte, parametros e k de referencia. `modelos_numericos.json` guarda os centros e as escalas para reconstruir as atribuicoes sem carregar objetos Python de origem desconhecida. `localizacao_12_prioridades.csv` preserva a identificacao das prioridades de desenvolvimento, sem consultar status R4.

As duas representacoes sao analises complementares, nao dois testes independentes. Nao comparar silhouette entre espacos como uma prova de superioridade biologica. A avaliacao de placas nao distingue automaticamente composicao de compostos de efeitos tecnicos. As 18 condicoes incompletas continuam fora desta base; nenhuma imputacao foi aplicada.

**Fontes tecnicas primarias:**
- [K-means e limites geometricos](https://scikit-learn.org/stable/modules/clustering.html#k-means)
- [Parametros KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
- [Silhouette](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html)
- [Indice de Rand ajustado](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html)
- [RobustScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html)

A documentacao define os algoritmos; as escolhas do protocolo e as interpretacoes sao do presente estudo. Rotulos farmacologicos e classificacao supervisionada nao sao pre-requisitos para esta etapa de descoberta.